In [19]:
%load_ext cython

The cython extension is already loaded. To reload it, use:
  %reload_ext cython


In [1]:
# Cell 1: Enhanced Environment Setup
%load_ext cython
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from typing import Dict, Any, Optional
import warnings
import json
import time
from pathlib import Path
import threading
import queue
import concurrent.futures
import requests
import hashlib
warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("=== ENHANCED LLM-INTEGRATED ENVIRONMENT ===")

# Enhanced Configuration Memory with LLM Feedback
class LLMEnhancedConfigurationMemory:
    def __init__(self):
        self.successful_configs = []
        self.failed_configs = []
        self.performance_history = []
        self.best_ever_config = None
        self.best_ever_hv = 0
        self.llm_suggestions = []
        self.operator_performance = {}
        self.parameter_regions = {}
        self.instance_features = {}  # Store instance characteristics
        
    def add_result(self, config, success, hv_improvement, runtime, hv, operator_strategy=None, instance_features=None):
        result = {
            'config': config,
            'success': success,
            'hv_improvement': hv_improvement,
            'runtime': runtime,
            'hv': hv,
            'timestamp': time.time(),
            'operator_strategy': operator_strategy,
            'instance_features': instance_features
        }
        
        # Track operator performance
        if operator_strategy:
            op_key = '_'.join(operator_strategy)
            if op_key not in self.operator_performance:
                self.operator_performance[op_key] = []
            self.operator_performance[op_key].append(hv_improvement)
        
        # Track parameter region
        region_key = (config['alpha']//5, int(config['kappa']*100)//1, config['L'])
        if region_key not in self.parameter_regions:
            self.parameter_regions[region_key] = []
        self.parameter_regions[region_key].append(hv_improvement)
        
        # Track best ever configuration
        if hv > self.best_ever_hv:
            self.best_ever_hv = hv
            self.best_ever_config = config.copy()
            
        if success and hv_improvement > 0:
            self.successful_configs.append(result)
        else:
            self.failed_configs.append(result)
        self.performance_history.append(result)
    
    def get_best_operators(self, top_n=3):
        """Get top performing operator strategies"""
        if not self.operator_performance:
            return ['swap', 'greedy_add', 'mutation']
        
        avg_performance = {}
        for op, improvements in self.operator_performance.items():
            avg_performance[op] = np.mean(improvements)
        
        sorted_ops = sorted(avg_performance.items(), key=lambda x: x[1], reverse=True)
        return [op.split('_') for op, _ in sorted_ops[:top_n]]
    
    def get_best_regions(self, top_n=3):
        """Get top performing parameter regions"""
        if not self.parameter_regions:
            return [(30, 10, 4)]
        
        avg_performance = {}
        for region, improvements in self.parameter_regions.items():
            avg_performance[region] = np.mean(improvements)
        
        sorted_regions = sorted(avg_performance.items(), key=lambda x: x[1], reverse=True)
        return sorted_regions[:top_n]

# Initialize global memory
config_memory = LLMEnhancedConfigurationMemory()

# Load the true reference Pareto front
def load_true_reference_pareto_front(reference_file="./result250.txt"):
    """Load the reference results and extract the true Pareto front"""
    try:
        data = np.loadtxt(reference_file)
        data_unique = np.unique(data, axis=0)
        
        def is_pareto_efficient(points):
            """Find the indices of Pareto-efficient points (maximization)"""
            is_efficient = np.ones(points.shape[0], dtype=bool)
            for i, c in enumerate(points):
                if is_efficient[i]:
                    is_efficient[is_efficient] = np.any(points[is_efficient]>=c, axis=1)
                    is_efficient[i] = True
            return is_efficient
        
        pareto_mask = is_pareto_efficient(data_unique)
        pareto_points = data_unique[pareto_mask]
        
        return {
            'all_solutions': data,
            'unique_solutions': data_unique,
            'pareto_solutions': pareto_points,
            'num_pareto': len(pareto_points)
        }
    except Exception as e:
        print(f"Error loading reference file: {e}")
        return None

# Calculate hypervolume function with improved precision
def calculate_hypervolume_2d(pareto_front, reference_point=None):
    """Calculate 2D hypervolume of Pareto front with improved precision"""
    if len(pareto_front) == 0:
        return 0.0
    if reference_point is None:
        reference_point = np.array([0.0, 0.0])
    sorted_front = pareto_front[np.argsort(pareto_front[:, 0])]
    hypervolume = 0.0
    prev_x = reference_point[0]
    for point in sorted_front:
        width = point[0] - prev_x
        height = point[1] - reference_point[1]
        if width > 0 and height > 0:
            hypervolume += width * height
        prev_x = point[0]
    return hypervolume

# Load data
print("Loading true reference Pareto front...")
true_reference = load_true_reference_pareto_front()

if true_reference:
    true_reference_hv = calculate_hypervolume_2d(true_reference['pareto_solutions'])
    # Set a more achievable target (3% improvement for more ambitious goal)
    target_hv = true_reference_hv * 1.03
    print(f"✓ True Reference HV: {true_reference_hv:,.0f}")
    print(f"✓ Target HV (3% improvement): {target_hv:,.0f}")
else:
    print("⚠ Could not load true reference data, using default values")
    true_reference_hv = 97640877
    target_hv = true_reference_hv * 1.03

# Initialize results summary
results_summary = {
    'True_Reference': {
        'hv': true_reference_hv,
        'time': 0.12,
        'num_solutions': true_reference['num_pareto'] if true_reference else 236,
        'efficiency': true_reference_hv / 0.12
    }
}

print("\n=== LLM-ENHANCED ENVIRONMENT READY ===")
print(f"True Reference HV: {true_reference_hv:,.0f}")
print(f"Target HV: {target_hv:,.0f}")
print("Ready for LLM-enhanced optimization...")

=== ENHANCED LLM-INTEGRATED ENVIRONMENT ===
Loading true reference Pareto front...
✓ True Reference HV: 97,640,877
✓ Target HV (3% improvement): 100,570,103

=== LLM-ENHANCED ENVIRONMENT READY ===
True Reference HV: 97,640,877
Target HV: 100,570,103
Ready for LLM-enhanced optimization...


In [2]:
# Cell 2: Fixed LLaMA-3 Interface with Caching and Timeout Management
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import json
import warnings
from datetime import datetime
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, asdict
from pathlib import Path
import subprocess
import sys
import os
import requests
import hashlib

warnings.filterwarnings('ignore')

class FixedLLaMA3Interface:
    """Fixed LLaMA-3 interface with caching, timeout management, and response validation"""
    
    def __init__(self, model_path="llama3:latest", temperature=0.7, max_tokens=500):
        self.model_path = model_path
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.cache = {}  # Cache responses to avoid repeated calls
        self.call_count = 0
        self.successful_calls = 0
        self.failed_calls = 0
        self.connection_status = None
        self.response_cache = {}  # Hash-based caching
        self._verify_connection()
        
    def _verify_connection(self):
        """Verify if LLaMA-3 is accessible"""
        print("🔍 Verifying LLaMA-3 connection...")
        
        # Test Ollama connection with shorter timeout
        try:
            response = requests.get('http://localhost:11434/api/tags', timeout=3)
            if response.status_code == 200:
                models = response.json().get('models', [])
                model_names = [model['name'] for model in models]
                
                if self.model_path in model_names:
                    self.connection_status = "ollama_connected"
                    print(f"✅ Ollama connected with {self.model_path} available")
                    return True
                else:
                    print(f"⚠️ Ollama connected but {self.model_path} not found")
                    print(f"Available models: {model_names}")
                    self.connection_status = "ollama_no_model"
                    return False
            else:
                print(f"❌ Ollama API returned status {response.status_code}")
                self.connection_status = "ollama_failed"
        except Exception as e:
            print(f"❌ Ollama connection failed: {e}")
            self.connection_status = "ollama_unavailable"
        
        print("❌ No LLaMA-3 connection available - using rule-based optimization")
        return False
    
    def _call_ollama(self, prompt):
        """Call local Ollama API for LLaMA-3 with improved timeout and error handling"""
        try:
            response = requests.post(
                'http://localhost:11434/api/generate',
                json={
                    'model': self.model_path,
                    'prompt': prompt,
                    'stream': False,
                    'options': {
                        'temperature': self.temperature,
                        'num_predict': self.max_tokens,
                        'timeout': 5  # Much shorter timeout
                    }
                },
                timeout=10  # Much shorter timeout
            )
            if response.status_code == 200:
                self.successful_calls += 1
                return response.json()['response']
            else:
                self.failed_calls += 1
                print(f"⚠️ Ollama API error: {response.status_code}")
                return None
        except Exception as e:
            self.failed_calls += 1
            print(f"⚠️ Error calling Ollama: {e}")
            return None
    
    def _get_cache_key(self, prompt):
        """Generate cache key for prompt"""
        return hashlib.md5(prompt.encode()).hexdigest()
    
    def generate_config_suggestion(self, current_state, performance_history, adaptive_bounds):
        """Generate configuration suggestion using LLaMA-3 with caching and fallback"""
        
        # Create cache key
        state_key = f"{current_state.get('hv', 0)}_{current_state.get('alpha', 30)}_{current_state.get('kappa', 0.10)}"
        cache_key = self._get_cache_key(state_key)
        
        # Check cache first
        if cache_key in self.response_cache:
            print("📋 Using cached LLM response")
            return self.response_cache[cache_key]
        
        # If LLM is not available, use rule-based approach
        if self.connection_status != "ollama_connected":
            return self._generate_rule_based_config(current_state, adaptive_bounds)
        
        # Try LLM with shorter timeout
        try:
            # Calculate precise gap to dominance
            current_hv = current_state.get('hv', 0)
            reference_hv = current_state.get('reference_hv', 97640877)
            target_hv = reference_hv * 1.03
            gap_to_target = target_hv - current_hv
            
            # Get best performing operators
            best_operators = config_memory.get_best_operators(3)
            
            # Enhanced prompt for LLaMA-3 with explicit operator names and constraints
            prompt = f"""
You are an expert optimization algorithm designer specializing in multi-objective knapsack problems.

CURRENT STATUS:
- Current Hypervolume (HV): {current_hv:,.0f}
- Reference HV: {reference_hv:,.0f}
- Target HV (3% improvement): {target_hv:,.0f}
- Gap to target: {gap_to_target:,.0f} HV points
- Current parameters: α={current_state.get('alpha', 30)}, κ={current_state.get('kappa', 0.10):.3f}, L={current_state.get('L', 4)}

PARAMETER CONSTRAINTS:
- alpha: {adaptive_bounds['alpha'][0]} to {adaptive_bounds['alpha'][1]}
- kappa: {adaptive_bounds['kappa'][0]:.3f} to {adaptive_bounds['kappa'][1]:.3f}
- L: {adaptive_bounds['L'][0]} to {adaptive_bounds['L'][1]}

TASK:
Generate a NEW configuration that has NOT been tried before.
Focus on achieving the target hypervolume.
Use ONLY these operators: swap, greedy_add, mutation, local_search

RESPONSE FORMAT (JSON only):
{{
    "alpha": <integer>,
    "kappa": <float with 3 decimals>,
    "L": <integer>,
    "operator_strategy": [<list of 2-3 operators from: swap, greedy_add, mutation, local_search>],
    "reasoning": "<brief explanation>"
}}
"""
            
            response = self._call_ollama(prompt)
            
            if response:
                try:
                    # Extract JSON from response
                    start_idx = response.find('{')
                    end_idx = response.rfind('}') + 1
                    if start_idx != -1 and end_idx != -1:
                        json_str = response[start_idx:end_idx]
                        config = json.loads(json_str)
                        
                        # Validate parameters
                        alpha = config.get('alpha', 30)
                        kappa = config.get('kappa', 0.10)
                        L = config.get('L', 4)
                        
                        # Apply bounds
                        alpha = max(adaptive_bounds['alpha'][0], min(adaptive_bounds['alpha'][1], alpha))
                        kappa = max(adaptive_bounds['kappa'][0], min(adaptive_bounds['kappa'][1], kappa))
                        L = max(adaptive_bounds['L'][0], min(adaptive_bounds['L'][1], L))
                        
                        # Validate operators
                        valid_operators = ['swap', 'greedy_add', 'mutation', 'local_search']
                        strategy = config.get('operator_strategy', ['swap', 'greedy_add'])
                        valid_strategy = [op for op in strategy if op in valid_operators]
                        if not valid_strategy:
                            valid_strategy = ['swap', 'greedy_add']
                        
                        result_config = {
                            'alpha': alpha,
                            'kappa': kappa,
                            'L': L,
                            'operator_strategy': valid_strategy,
                            'reasoning': config.get('reasoning', 'LLM generated')
                        }
                        
                        # Cache the response
                        self.response_cache[cache_key] = result_config
                        return result_config
                        
                except json.JSONDecodeError:
                    print("⚠️ Could not parse JSON from LLM response")
            
        except Exception as e:
            print(f"⚠️ LLM call failed: {e}")
        
        # Fallback to rule-based
        return self._generate_rule_based_config(current_state, adaptive_bounds)
    
    def _generate_rule_based_config(self, current_state, adaptive_bounds):
        """Generate rule-based configuration when LLM fails"""
        
        current_hv = current_state.get('hv', 0)
        reference_hv = current_state.get('reference_hv', 97640877)
        target_hv = reference_hv * 1.03
        gap_to_target = target_hv - current_hv
        
        # Use best performing regions
        best_regions = config_memory.get_best_regions(3)
        best_operators = config_memory.get_best_operators(3)
        
        # More conservative parameter adjustment to avoid runtime explosion
        if gap_to_target > 2000000:  # Large gap
            new_config = {
                'alpha': min(50, max(25, current_state.get('alpha', 30) + 10)),
                'kappa': min(0.20, max(0.05, current_state.get('kappa', 0.10) + 0.05)),
                'L': min(8, max(3, current_state.get('L', 4) + 2)),
                'small_value': 1e-7
            }
        elif gap_to_target > 1000000:  # Medium gap
            new_config = {
                'alpha': min(50, max(25, current_state.get('alpha', 30) + 5)),
                'kappa': min(0.20, max(0.05, current_state.get('kappa', 0.10) + 0.03)),
                'L': min(8, max(3, current_state.get('L', 4) + 1)),
                'small_value': 1e-7
            }
        else:  # Small gap - fine-tune
            new_config = {
                'alpha': min(50, max(25, current_state.get('alpha', 30) + np.random.randint(-1, 2))),
                'kappa': min(0.20, max(0.05, current_state.get('kappa', 0.10) + np.random.uniform(-0.005, 0.005))),
                'L': min(8, max(3, current_state.get('L', 4) + np.random.randint(-1, 2))),
                'small_value': 1e-7
            }
        
        # Apply bounds
        new_config['alpha'] = max(adaptive_bounds['alpha'][0], min(adaptive_bounds['alpha'][1], new_config['alpha']))
        new_config['kappa'] = max(adaptive_bounds['kappa'][0], min(adaptive_bounds['kappa'][1], new_config['kappa']))
        new_config['L'] = max(adaptive_bounds['L'][0], min(adaptive_bounds['L'][1], new_config['L']))
        
        # Select best operators
        operator_strategy = best_operators[0] if best_operators else ['swap', 'greedy_add', 'mutation']
        
        return {
            'alpha': new_config['alpha'],
            'kappa': new_config['kappa'],
            'L': new_config['L'],
            'operator_strategy': operator_strategy,
            'reasoning': 'Rule-based configuration due to LLM unavailability'
        }

# Initialize fixed LLM interface with correct model path
try:
    fixed_llm_interface = FixedLLaMA3Interface(model_path="llama3:latest", temperature=0.7)
    
    # Show status
    if fixed_llm_interface.connection_status == "ollama_connected":
        print("✅ LLaMA-3 is working properly!")
        llm_working = True
    else:
        print("⚠️ LLaMA-3 is not working properly, using fallback")
        llm_working = False
        
except Exception as e:
    print(f"❌ Failed to initialize LLaMA-3: {e}")
    llm_working = False
    fixed_llm_interface = None

# Enhanced parameter bounds (more conservative to avoid runtime issues)
ENHANCED_PARAMETER_BOUNDS = {
    'alpha': (25, 50),    # More conservative range
    'kappa': (0.05, 0.20), # More conservative range  
    'L': (3, 8),          # More conservative range
    'small_value': (1e-9, 1e-6)
}

print("Enhanced LLM Integration Ready!")
print(f"Parameter bounds: {ENHANCED_PARAMETER_BOUNDS}")

🔍 Verifying LLaMA-3 connection...
✅ Ollama connected with llama3:latest available
✅ LLaMA-3 is working properly!
Enhanced LLM Integration Ready!
Parameter bounds: {'alpha': (25, 50), 'kappa': (0.05, 0.2), 'L': (3, 8), 'small_value': (1e-09, 1e-06)}


In [4]:
%%cython
"""
Optimized MOACP Implementation with LLM Integration and Runtime Controls
- Enhanced performance with conservative parameters
- Improved parameter handling with bounds checking
- Better operator selection with validation
- Optimized for achieving dominance within runtime constraints
"""

from libc.stdlib cimport malloc, free, srand, rand
from libc.string cimport memset
from libc.math cimport exp
import numpy as np
import time
import matplotlib.pyplot as plt

# Optimized MHRE Parameters (more conservative)
cdef int OPTIMIZED_ALPHA = 35
cdef float OPTIMIZED_KAPPA = 0.12
cdef int OPTIMIZED_L = 5
cdef float OPTIMIZED_SMALL_VALUE = 1e-7

# Baseline Parameters
cdef int BASELINE_ALPHA = 10
cdef float BASELINE_KAPPA = 0.05
cdef int BASELINE_L = 5
cdef float BASELINE_SMALL_VALUE = 1e-7

# Enhanced bounds (more conservative)
cdef int MAX_ALPHA = 50
cdef float MAX_KAPPA = 0.20
cdef int MAX_L = 8

# Structs
cdef struct ind:
    int nombr_nonpris
    int nombr
    int rank
    float fitnessbest
    float fitness
    int explored
    double *f
    double *capa
    double *v
    int *d
    int *Items

cdef struct pop:
    int size
    int maxsize
    ind **ind_array

# Globals
cdef int NBITEMS = 250
cdef int ni = 250
cdef int L = 5
cdef double LARGE = 10e50
cdef float smallValue = 0.0000001
cdef double kappa = 0.05
cdef int alpha = 10
cdef int paretoIni = 28000

cdef int nf = 2
cdef double *capacities = NULL
cdef int **weights = NULL
cdef int **profits = NULL
cdef double *vector_weight = NULL
cdef double max_bound = 0.0
cdef double **OBJ_Weights = NULL
cdef int nombreLIGNE = 0
cdef int nextLn = 0
cdef int inv = 0
cdef int OBJ_Weights_lines = 0

def seed(int x):
    srand(x)

cdef int irand(int range_val):
    return rand() % range_val

cdef void *chk_malloc(size_t size):
    cdef void *return_value = malloc(size)
    if return_value == NULL:
        raise MemoryError("Out of memory.")
    memset(return_value, 0, size)
    return return_value

cdef pop *create_pop(int maxsize, int nf):
    cdef int i
    cdef pop *pp = <pop *>chk_malloc(sizeof(pop))
    pp.size = 0
    pp.maxsize = maxsize
    pp.ind_array = <ind **>chk_malloc(maxsize * sizeof(void*))
    for i in range(maxsize):
        pp.ind_array[i] = NULL
    return pp

cdef ind *create_ind(int nf):
    cdef int i
    cdef ind *p_ind = <ind *>chk_malloc(sizeof(ind))
    p_ind.nombr_nonpris = 0
    p_ind.nombr = 0
    p_ind.rank = 0
    p_ind.fitnessbest = -1.0
    p_ind.fitness = -1.0
    p_ind.explored = 0
    p_ind.f = <double *>chk_malloc(nf * sizeof(double))
    p_ind.capa = <double *>chk_malloc(nf * sizeof(double))
    p_ind.v = <double *>chk_malloc(nf * sizeof(double))
    p_ind.d = <int *>chk_malloc(ni * sizeof(int))
    p_ind.Items = <int *>chk_malloc(ni * sizeof(int))
    for i in range(ni):
        p_ind.Items[i] = 0
        p_ind.d[i] = 0
    for i in range(nf):
        p_ind.f[i] = 0.0
        p_ind.capa[i] = 0.0
        p_ind.v[i] = 0.0
    return p_ind

cdef ind *ind_copy(ind *i):
    cdef ind *p_ind = create_ind(nf)
    cdef int k
    p_ind.nombr_nonpris = i.nombr_nonpris
    p_ind.nombr = i.nombr
    p_ind.rank = i.rank
    p_ind.fitnessbest = i.fitnessbest
    p_ind.fitness = i.fitness
    p_ind.explored = i.explored
    for k in range(nf):
        p_ind.f[k] = i.f[k]
        p_ind.v[k] = i.v[k]
        p_ind.capa[k] = i.capa[k]
    for k in range(ni):
        p_ind.d[k] = i.d[k]
        p_ind.Items[k] = i.Items[k]
    return p_ind

cdef void free_ind(ind *p_ind):
    if p_ind != NULL:
        free(p_ind.d)
        free(p_ind.f)
        free(p_ind.capa)
        free(p_ind.v)
        free(p_ind.Items)
        free(p_ind)

cdef void complete_free_pop(pop *pp):
    cdef int i
    if pp != NULL:
        if pp.ind_array != NULL:
            for i in range(pp.size):
                if pp.ind_array[i] != NULL:
                    free_ind(pp.ind_array[i])
                    pp.ind_array[i] = NULL
            free(pp.ind_array)
        free(pp)

cdef void cleanup_globals():
    global capacities, weights, profits, vector_weight, OBJ_Weights, OBJ_Weights_lines, nf, ni
    if capacities != NULL:
        free(capacities)
        capacities = NULL
    if weights != NULL:
        for i in range(nf):
            if weights[i] != NULL:
                free(weights[i])
        free(weights)
        weights = NULL
    if profits != NULL:
        for i in range(nf):
            if profits[i] != NULL:
                free(profits[i])
        free(profits)
        profits = NULL
    if vector_weight != NULL:
        free(vector_weight)
        vector_weight = NULL
    if OBJ_Weights != NULL:
        for i in range(nf):
            if OBJ_Weights[i] != NULL:
                free(OBJ_Weights[i])
        free(OBJ_Weights)
        OBJ_Weights = NULL
    OBJ_Weights_lines = 0
    nf = 0
    ni = 0

cdef int non_dominated(ind *p_ind_a, ind *p_ind_b):
    cdef int i
    cdef int a_is_good = -1
    cdef int equal = 1
    for i in range(nf):
        if p_ind_a.f[i] > p_ind_b.f[i]:
            a_is_good = 1
        if p_ind_a.f[i] != p_ind_b.f[i]:
            equal = 0
    if equal:
        return 0
    return a_is_good

cdef double calcAddEpsIndicator(ind *p_ind_a, ind *p_ind_b):
    global max_bound
    cdef int i
    cdef double eps
    cdef double temp_eps
    if max_bound == 0.0:
        max_bound = 1e-8
    eps = (p_ind_a.v[0]/max_bound)-(p_ind_b.v[0]/max_bound)
    for i in range(1, nf):
        temp_eps = (p_ind_a.v[i]/max_bound)-(p_ind_b.v[i]/max_bound)
        if temp_eps > eps:
            eps = temp_eps
    return eps

cdef void init_fitness(ind *x):
    x.fitness = 0.0

cdef void update_fitness(ind *x, double I):
    x.fitness -= exp(-I / kappa)

cdef double update_fitness_return(double f, double I):
    return f - exp(-I / kappa)

cdef int delete_fitness(ind *x, double I):
    x.fitness += exp(-I / kappa)
    return 0

cdef void compute_ind_fitness(ind *x, pop *SP):
    cdef int j
    init_fitness(x)
    for j in range(SP.size):
        if SP.ind_array[j] != x:
            update_fitness(x, calcAddEpsIndicator(SP.ind_array[j], x))

cdef void compute_all_fitness(pop *SP):
    cdef int i
    for i in range(SP.size):
        compute_ind_fitness(SP.ind_array[i], SP)

cdef void loadMOKP(char *filename):
    global nf, ni, capacities, weights, profits
    cdef int i, f
    with open(filename.decode(), "r") as source:
        _nf, _ni = [int(x) for x in source.readline().split()]
        nf = _nf
        ni = _ni
        capacities = <double *>chk_malloc(nf * sizeof(double))
        weights = <int **>chk_malloc(nf * sizeof(void*))
        profits = <int **>chk_malloc(nf * sizeof(void*))
        for f in range(nf):
            capacities[f] = float(source.readline().strip())
            weights[f] = <int *>chk_malloc(ni * sizeof(int))
            profits[f] = <int *>chk_malloc(ni * sizeof(int))
            for i in range(ni):
                source.readline()  # item index (ignore)
                weights[f][i] = int(source.readline().strip())
                profits[f][i] = int(source.readline().strip())

cdef void read_weights_file(char *filename):
    global OBJ_Weights, nombreLIGNE, nf, OBJ_Weights_lines
    cdef int i, j, nlines
    with open(filename.decode(), "r") as f:
        lines = [line for line in f if line.strip()]
    nlines = len(lines)
    OBJ_Weights = <double **>chk_malloc(nf * sizeof(void*))
    for i in range(nf):
        OBJ_Weights[i] = <double *>chk_malloc(nlines * sizeof(double))
    for i, line in enumerate(lines):
        vals = line.strip().split()
        for j in range(nf):
            OBJ_Weights[j][i] = float(vals[j])
    nombreLIGNE = nlines - 1
    OBJ_Weights_lines = nlines

cdef void dynamic_weight_allpop():
    global vector_weight, OBJ_Weights, nombreLIGNE, nf, nextLn
    cdef int i
    if vector_weight == NULL:
        vector_weight = <double *>chk_malloc(nf * sizeof(double))
    for i in range(nf):
        vector_weight[i] = OBJ_Weights[i][nextLn]
    if nextLn == nombreLIGNE:
        nextLn = 0
    else:
        nextLn += 1

cdef void choose_weight():
    dynamic_weight_allpop()

cdef void random_init_ind(ind *x):
    cdef int j, r, tmp
    for j in range(ni):
        x.d[j] = j
    for j in range(ni):
        r = irand(ni)
        tmp = x.d[r]
        x.d[r] = x.d[j]
        x.d[j] = tmp

cdef void evaluate(ind *x):
    cdef int j, l, k, faisable
    x.nombr = 0
    x.nombr_nonpris = 0
    for j in range(nf):
        x.capa[j] = 0.0
        x.f[j] = 0.0
    for j in range(ni):
        l = 0
        faisable = 1
        while l < nf and faisable == 1:
            if x.capa[l] + weights[l][x.d[j]] > capacities[l]:
                faisable = 0
            l += 1
        if faisable == 1:
            for k in range(nf):
                x.capa[k] += weights[k][x.d[j]]
                x.f[k] += profits[k][x.d[j]]
            x.Items[x.d[j]] = 1
            x.nombr += 1
        else:
            x.Items[x.d[j]] = 0
            x.nombr_nonpris += 1

cdef void P_init_pop(pop *SP, pop *Sarchive, int alpha):
    cdef int i, x, tmp, t
    t = max(alpha, Sarchive.size)
    cdef int* shuffle = <int *>chk_malloc(t * sizeof(int))
    for i in range(t):
        shuffle[i] = i
    for i in range(t):
        x = irand(alpha)
        tmp = shuffle[i]
        shuffle[i] = shuffle[x]
        shuffle[x] = tmp
    SP.size = alpha
    if Sarchive.size > alpha:
        for i in range(alpha):
            SP.ind_array[i] = ind_copy(Sarchive.ind_array[shuffle[i]])
    else:
        for i in range(alpha):
            if shuffle[i] < Sarchive.size:
                SP.ind_array[i] = ind_copy(Sarchive.ind_array[shuffle[i]])
            else:
                SP.ind_array[i] = create_ind(nf)
                random_init_ind(SP.ind_array[i])
                evaluate(SP.ind_array[i])
    free(shuffle)

cdef int extractPtoArchive(pop *P, pop *archive):
    cdef int i, j, dom, t, convergence_rate
    t = archive.size + P.size
    archiveAndP = create_pop(t, nf)
    convergence_rate = 0
    for i in range(archive.size):
        archiveAndP.ind_array[i] = archive.ind_array[i]
    for i in range(P.size):
        archiveAndP.ind_array[i + archive.size] = ind_copy(P.ind_array[i])
    archiveAndP.size = t
    archive.size = 0
    for i in range(t):
        for j in range(t):
            if i != j:
                dom = non_dominated(archiveAndP.ind_array[i], archiveAndP.ind_array[j])
                if dom == -1 or (dom == 0 and i > j):
                    break
        else:
            archive.ind_array[archive.size] = ind_copy(archiveAndP.ind_array[i])
            archive.size += 1
            if i >= t - P.size:
                convergence_rate += 1
    complete_free_pop(archiveAndP)
    return convergence_rate

cdef double calcMaxbound(pop *SP, int size):
    global max_bound
    cdef int i, j
    SP.size = size
    cdef double max_b = SP.ind_array[0].v[0]
    for i in range(SP.size):
        for j in range(nf):
            if max_b < SP.ind_array[i].v[j]:
                max_b = SP.ind_array[i].v[j]
    if max_b == 0.0:
        max_b = 1e-8
    max_bound = max_b
    return max_b

cdef void calcul_weight(pop *SP, int size):
    cdef int i, j
    for i in range(SP.size):
        for j in range(nf):
            SP.ind_array[i].v[j] = SP.ind_array[i].f[j] * vector_weight[j]

cdef int compute_fitness_and_select(pop *SP, ind *x, int size):
    cdef int i, worst
    cdef double worst_fit, fit_tmp
    SP.size = size
    x.fitness = 0
    compute_ind_fitness(x, SP)
    worst_fit = x.fitness
    worst = -1
    for i in range(SP.size):
        fit_tmp = update_fitness_return(SP.ind_array[i].fitness, calcAddEpsIndicator(x, SP.ind_array[i]))
        if fit_tmp > worst_fit:
            worst = i
            worst_fit = fit_tmp
    fit_tmp = x.fitness
    if worst == -1:
        return -1
    else:
        for i in range(SP.size):
            delete_fitness(SP.ind_array[i], calcAddEpsIndicator(SP.ind_array[worst], SP.ind_array[i]))
            update_fitness(SP.ind_array[i], calcAddEpsIndicator(x, SP.ind_array[i]))
        delete_fitness(x, calcAddEpsIndicator(SP.ind_array[worst], x))
        free_ind(SP.ind_array[worst])
        SP.ind_array[worst] = ind_copy(x)
        if fit_tmp - worst_fit > smallValue:
            return worst
        else:
            return -1

# Enhanced local search with runtime controls
cdef void enhanced_local_search(pop *SP, pop *Sarchive, int size, double time_budget):
    cdef ind *x
    cdef int i, j, r, t, k, l, v, sol, mino, mp, maxp, consistant, pos, stop, convergence, ii, tmp_pris, tmp_nonpris, taille, feasible, tv, IM
    cdef int* remplace = <int *>chk_malloc(L * sizeof(int))
    cdef double start_time = time.time()
    cdef double current_time
    
    SP.size = size
    extractPtoArchive(SP, Sarchive)
    
    # Enhanced early termination with convergence check
    cdef int no_improvement_count = 0
    cdef double best_hv = 0.0
    
    while (time.time() - start_time) < time_budget * 0.8 and no_improvement_count < 2:
        convergence = 0
        for i in range(SP.size):
            current_time = time.time()
            
            if (current_time - start_time) > time_budget * 0.8:
                break
                
            if not SP.ind_array[i].explored:
                x = ind_copy(SP.ind_array[i])
                j = 0
                while j < x.nombr and (current_time - start_time) < time_budget * 0.8:
                    for l in range(L):
                        remplace[l] = 0
                    while True:
                        mino = irand(ni)
                        if x.Items[mino] == 1:
                            break
                    x.Items[mino] = 0
                    x.nombr -= 1
                    x.nombr_nonpris += 1
                    for r in range(nf):
                        x.capa[r] -= weights[r][mino]
                        x.f[r] -= profits[r][mino]
                    IM = 0
                    taille = 0
                    while IM < L and (current_time - start_time) < time_budget * 0.8:
                        while True:
                            maxp = irand(ni)
                            if x.Items[maxp] == 0:
                                break
                        if maxp != mino:
                            consistant = 1
                            r = 0
                            while r < nf and consistant == 1:
                                if x.capa[r] + weights[r][maxp] > capacities[r]:
                                    consistant = 0
                                r += 1
                            if consistant == 1:
                                feasible = 1
                                r = 0
                                while r < taille and feasible:
                                    if maxp == remplace[r]:
                                        feasible = 0
                                    r += 1
                                if feasible == 1:
                                    remplace[taille] = maxp
                                    taille += 1
                                    x.Items[maxp] = 1
                                    x.nombr_nonpris -= 1
                                    x.nombr += 1
                                    for r in range(nf):
                                        x.capa[r] += weights[r][maxp]
                                        x.f[r] += profits[r][maxp]
                        IM += 1
                    for tv in range(nf):
                        x.v[tv] = x.f[tv] * vector_weight[tv]
                    max_bound = calcMaxbound(SP, SP.size)
                    sol = compute_fitness_and_select(SP, x, SP.size)
                    if sol != -1:
                        j = x.nombr + 1
                        if sol > i and i + 1 < SP.size:
                            y = SP.ind_array[i + 1]
                            SP.ind_array[i + 1] = SP.ind_array[sol]
                            SP.ind_array[sol] = y
                            i += 1
                        break
                    elif sol == -1:
                        x.Items[mino] = 1
                        x.nombr_nonpris -= 1
                        x.nombr += 1
                        for r in range(nf):
                            x.capa[r] += weights[r][mino]
                            x.f[r] += profits[r][mino]
                        if taille >= 1:
                            for r in range(taille):
                                x.Items[remplace[r]] = 0
                                x.nombr -= 1
                                x.nombr_nonpris += 1
                                for t in range(nf):
                                    x.capa[t] -= weights[t][remplace[r]]
                                    x.f[t] -= profits[t][remplace[r]]
                                    x.v[t] = x.f[t] * vector_weight[t]
                    j += 1
                tmp_pris = x.nombr
                tmp_nonpris = x.nombr_nonpris
                free_ind(x)
                if j == tmp_pris:
                    SP.ind_array[i].explored = 1
        convergence = extractPtoArchive(SP, Sarchive)
        if not convergence:
            no_improvement_count += 1
        else:
            no_improvement_count = 0
    free(remplace)

# Greedy add operator with runtime controls
cdef void greedy_add_operator(pop *SP, pop *Sarchive, int size, double time_budget):
    cdef int i, best_item, r
    cdef double best_gain, gain
    cdef ind *x
    cdef double start_time = time.time()
    
    SP.size = size
    extractPtoArchive(SP, Sarchive)
    
    for i in range(SP.size):
        if (time.time() - start_time) > time_budget * 0.8:
            break
            
        x = ind_copy(SP.ind_array[i])
        best_item = -1
        best_gain = -1.0
        for j in range(ni):
            if (time.time() - start_time) > time_budget * 0.8:
                break
                
            if x.Items[j] == 0:
                feasible = 1
                for r in range(nf):
                    if x.capa[r] + weights[r][j] > capacities[r]:
                        feasible = 0
                        break
                if feasible:
                    gain = 0.0
                    for r in range(nf):
                        gain += profits[r][j]
                    if gain > best_gain:
                        best_gain = gain
                        best_item = j
        if best_item != -1:
            x.Items[best_item] = 1
            x.nombr += 1
            x.nombr_nonpris -= 1
            for r in range(nf):
                x.capa[r] += weights[r][best_item]
                x.f[r] += profits[r][best_item]
            for r in range(nf):
                x.v[r] = x.f[r] * vector_weight[r]
            max_bound = calcMaxbound(SP, SP.size)
            compute_fitness_and_select(SP, x, SP.size)
        free_ind(x)

# Enhanced parameter configuration with bounds checking
cdef void set_optimized_parameters(
    bint use_optimized, 
    object custom_params=None, 
    object operator_strategy=None, 
    bint print_params=True
):
    global alpha, kappa, L, smallValue
    
    if custom_params is not None:
        # Apply enhanced bounds with strict checking
        alpha = min(MAX_ALPHA, max(25, int(custom_params.get('alpha', OPTIMIZED_ALPHA if use_optimized else BASELINE_ALPHA))))
        kappa = min(MAX_KAPPA, max(0.05, float(custom_params.get('kappa', OPTIMIZED_KAPPA if use_optimized else BASELINE_KAPPA))))
        L = min(MAX_L, max(3, int(custom_params.get('L', OPTIMIZED_L if use_optimized else BASELINE_L))))
        smallValue = float(custom_params.get('small_value', OPTIMIZED_SMALL_VALUE if use_optimized else BASELINE_SMALL_VALUE))
        if print_params:
            print(f"🚀 Optimized Parameters: alpha={alpha}, kappa={kappa:.3f}, L={L}, small={smallValue:.1e}")
    else:
        if use_optimized:
            alpha = OPTIMIZED_ALPHA
            kappa = OPTIMIZED_KAPPA
            L = OPTIMIZED_L
            smallValue = OPTIMIZED_SMALL_VALUE
            if print_params:
                print(f"🚀 Optimized Parameters: alpha={alpha}, kappa={kappa:.3f}, L={L}, small={smallValue:.1e}")
        else:
            alpha = BASELINE_ALPHA
            kappa = BASELINE_KAPPA
            L = BASELINE_L
            smallValue = BASELINE_SMALL_VALUE
            if print_params:
                print(f"📊 Baseline Parameters: alpha={alpha}, kappa={kappa:.3f}, L={L}, small={smallValue:.1e}")
    if operator_strategy is not None:
        print(f"   Operator strategy: {operator_strategy}")

# Enhanced main MOACP runner with LLM integration and runtime controls
def run_moacp_llm_enhanced(
    instance_file,
    weights_file,
    nbitems,
    num_objectives,
    use_optimized=True,
    num_runs=5,
    num_iterations=100,
    custom_params=None,
    operator_strategy=None,
    print_params=True,
    runtime_threshold=5.0,
    reference_hv=None,
    target_hv=None,
    llm_interface=None
):
    """
    Enhanced MOACP runner with LLM integration and runtime controls
    """
    global nf, ni, NBITEMS, alpha, paretoIni, L, nombreLIGNE, nextLn, inv, vector_weight
    global capacities, weights, profits, OBJ_Weights

    set_optimized_parameters(use_optimized, custom_params, operator_strategy, print_params)
    mode = "LLM-Enhanced" if use_optimized else "Baseline"
    NBITEMS = nbitems
    ni = nbitems
    nf = num_objectives
    paretoIni = 28000

    all_pareto_solutions = []
    run_times = []
    used_operators_log = []

    if print_params:
        print(f"\n {mode} MOACP: {num_runs} runs × {num_iterations} iterations")
        print(f" LLM-enhanced runtime threshold: {runtime_threshold:.2f}s per run")
        if reference_hv:
            print(f" Reference HV: {reference_hv:,.0f}")
        if target_hv:
            print(f" Target HV: {target_hv:,.0f}")

    total_start_time = time.time()

    for run in range(1, num_runs + 1):
        run_start_time = time.time()
        if print_params:
            print(f"   Run {run}/{num_runs}...", end=" ")

        nombreLIGNE = 0
        nextLn = 0
        inv = 0

        seed(run)
        loadMOKP(instance_file.encode())
        read_weights_file(weights_file.encode())

        vector_weight = <double *>chk_malloc(nf * sizeof(double))
        P = create_pop(paretoIni, nf)

        it = 0
        while it < num_iterations:
            iteration_start = time.time()
            
            solutions = create_pop(alpha, nf)
            archive = create_pop(paretoIni, nf)
            choose_weight()
            P_init_pop(solutions, P, alpha)
            extractPtoArchive(solutions, P)
            calcul_weight(solutions, alpha)
            calcMaxbound(solutions, alpha)
            compute_all_fitness(solutions)

            # Enhanced operator dispatch with LLM guidance and validation
            if operator_strategy is not None and isinstance(operator_strategy, (list, tuple)):
                # Enhanced time budget with focus on quality
                time_budget = runtime_threshold * 0.5 / len(operator_strategy)
                
                for op in operator_strategy:
                    op_start = time.time()
                    
                    # Skip operator if time budget exceeded
                    if (time.time() - iteration_start) > runtime_threshold * 0.7:
                        if print_params and it == 0 and run == 1:
                            print(f"   (Skipping {op} due to time constraint)")
                        continue
                    
                    used_operators_log.append(op)
                    
                    # Validate operator name
                    if op == "swap":
                        enhanced_local_search(solutions, archive, alpha, time_budget)
                    elif op == "greedy_add":
                        greedy_add_operator(solutions, archive, alpha, time_budget)
                    elif op == "mutation":
                        enhanced_local_search(solutions, archive, alpha, time_budget)
                    elif op == "local_search":
                        enhanced_local_search(solutions, archive, alpha, time_budget)
                    else:
                        if print_params and it == 0 and run == 1:
                            print(f"   (Warning: Operator '{op}' not recognized, using swap instead)")
                        enhanced_local_search(solutions, archive, alpha, time_budget)
            else:
                enhanced_local_search(solutions, archive, alpha, runtime_threshold * 0.5)

            extractPtoArchive(archive, P)
            it += 1
            complete_free_pop(solutions)
            complete_free_pop(archive)

        # Extract Pareto front for this run
        run_pareto = []
        for i in range(P.size):
            if P.ind_array[i] != NULL:
                obj1 = P.ind_array[i].f[0]
                obj2 = P.ind_array[i].f[1] if nf > 1 else 0
                run_pareto.append([obj1, obj2])

        all_pareto_solutions.extend(run_pareto)
        pareto_np = np.array(run_pareto)
        if pareto_np.shape[0] > 0:
            max_obj1 = np.max(pareto_np[:, 0])
            min_obj1 = np.min(pareto_np[:, 0])
            spread_obj1 = max_obj1 - min_obj1
            max_obj2 = np.max(pareto_np[:, 1])
            min_obj2 = np.min(pareto_np[:, 1])
            spread_obj2 = max_obj2 - min_obj2
        else:
            max_obj1 = min_obj1 = spread_obj1 = 0
            max_obj2 = min_obj2 = spread_obj2 = 0

        run_time = time.time() - run_start_time
        run_times.append(run_time)

        if print_params:
            status = "⚠️" if run_time > runtime_threshold else "✓"
            print(f"{status} {len(run_pareto)} solutions, {run_time:.2f}s")

        complete_free_pop(P)
        cleanup_globals()

    total_time = time.time() - total_start_time
    avg_time_per_run = total_time / num_runs if num_runs > 0 else 0.0

    if print_params:
        print(f" {mode} Complete: {len(all_pareto_solutions)} total solutions, {total_time:.2f}s total, {avg_time_per_run:.2f}s avg/run")

    return {
        'pareto_solutions': np.array(all_pareto_solutions) if all_pareto_solutions else np.array([]),
        'total_time': total_time,
        'avg_time_per_run': avg_time_per_run,
        'run_times': run_times,
        'mode': mode,
        'parameters': {
            'alpha': alpha,
            'kappa': kappa,
            'L': L,
            'small_value': smallValue
        },
        'num_solutions': len(all_pareto_solutions),
        'num_runs': num_runs,
        'num_iterations': num_iterations,
        'operator_strategy': operator_strategy,
        'used_operators_log': used_operators_log,
        'max_obj1': max_obj1,
        'min_obj1': min_obj1,
        'spread_obj1': spread_obj1,
        'max_obj2': max_obj2,
        'min_obj2': min_obj2,
        'spread_obj2': spread_obj2,
        'reference_hv': reference_hv,
        'target_hv': target_hv
    }

print(" LLM-Enhanced MOACP Implementation Ready!")

Content of stdout:
_cython_magic_373eba2fd2fb4e80b16966291b7e4c5fcbe2df43f0cd3aa44ff45ee531b335e3.c
C:\Users\Aziz il patal\.ipython\cython\_cython_magic_373eba2fd2fb4e80b16966291b7e4c5fcbe2df43f0cd3aa44ff45ee531b335e3.c(4907): warning C4244: '='ÿ: conversion de 'double' en 'float', perte possible de donn‚es
C:\Users\Aziz il patal\.ipython\cython\_cython_magic_373eba2fd2fb4e80b16966291b7e4c5fcbe2df43f0cd3aa44ff45ee531b335e3.c(4997): warning C4244: '='ÿ: conversion de 'double' en 'float', perte possible de donn‚es
C:\Users\Aziz il patal\.ipython\cython\_cython_magic_373eba2fd2fb4e80b16966291b7e4c5fcbe2df43f0cd3aa44ff45ee531b335e3.c(6007): warning C4244: '='ÿ: conversion de 'Py_ssize_t' en 'int', perte possible de donn‚es
C:\Users\Aziz il patal\.ipython\cython\_cython_magic_373eba2fd2fb4e80b16966291b7e4c5fcbe2df43f0cd3aa44ff45ee531b335e3.c(10043): warning C4244: '='ÿ: conversion de 'double' en 'float', perte possible de donn‚es
C:\Users\Aziz il patal\.ipython\cython\_cython_magic_373eba2f

In [5]:
# Cell 4: Enhanced Optimizer with Runtime Management
import numpy as np
import time
from typing import Dict, List, Any
import concurrent.futures
import threading

class FixedAdvancedLLMOptimizer:
    """Fixed optimizer with better runtime management and operator validation"""
    
    def __init__(self, llm_interface, config_memory):
        self.llm_interface = llm_interface
        self.config_memory = config_memory
        self.optimization_history = []
        self.best_result = None
        self.best_hv = 0
        self.dominance_achieved = False
        
    def optimize_instance(self, instance_file, weights_file, nbitems, num_objectives, 
                         reference_hv, target_hv, max_iterations=10):
        """
        Advanced optimization with LLM guidance and better runtime management
        """
        print(f"\n🚀 ADVANCED LLM-GUIDED OPTIMIZATION")
        print(f"Reference HV: {reference_hv:,.0f}")
        print(f"Target HV: {target_hv:,.0f}")
        print(f"Gap to close: {target_hv - reference_hv:,.0f}")
        print(f"LLM Status: {'✅ Working' if self.llm_interface and self.llm_interface.connection_status == 'ollama_connected' else '⚠️ Using fallback'}")
        
        # Initial configuration
        current_config = {
            'alpha': 35,
            'kappa': 0.12,
            'L': 5,
            'small_value': 1e-7
        }
        
        best_operators = ['swap', 'greedy_add', 'mutation']
        
        for iteration in range(max_iterations):
            print(f"\n🔄 Iteration {iteration + 1}/{max_iterations}")
            
            # Conservative runtime threshold to avoid timeout
            runtime_threshold = 5.0  # Fixed conservative threshold
            
            # Conservative number of runs and iterations
            num_runs = 6
            num_iterations = 100
            
            print(f"   Configuration: {num_runs} runs × {num_iterations} iterations, {runtime_threshold}s threshold")
            
            # Run with current configuration
            result = run_moacp_llm_enhanced(
                instance_file=instance_file,
                weights_file=weights_file,
                nbitems=nbitems,
                num_objectives=num_objectives,
                use_optimized=True,
                num_runs=num_runs,
                num_iterations=num_iterations,
                custom_params=current_config,
                operator_strategy=best_operators,
                print_params=True,
                runtime_threshold=runtime_threshold,
                reference_hv=reference_hv,
                target_hv=target_hv,
                llm_interface=self.llm_interface
            )
            
            # Calculate hypervolume
            hv = calculate_hypervolume_2d(result['pareto_solutions'])
            gap_to_target = target_hv - hv
            
            print(f"Result HV: {hv:,.0f}")
            print(f"Gap to target: {gap_to_target:,.0f}")
            
            # Update memory
            self.config_memory.add_result(
                current_config, 
                hv >= target_hv, 
                hv - reference_hv, 
                result['avg_time_per_run'], 
                hv,
                best_operators
            )
            
            # Check for dominance
            if hv >= target_hv:
                excess = hv - target_hv
                print(f"🎉 DOMINANCE ACHIEVED! Excess={excess:,.0f}")
                self.dominance_achieved = True
                self.best_result = result
                self.best_hv = hv
                return result, current_config, best_operators, True
            
            # Update best result
            if hv > self.best_hv:
                self.best_hv = hv
                self.best_result = result
                print(f"✓ New best result: {hv:,.0f}")
            
            # Get LLM suggestion for next iteration
            if self.llm_interface and self.llm_interface.connection_status == "ollama_connected" and iteration < max_iterations - 1:
                current_state = {
                    'hv': hv,
                    'reference_hv': reference_hv,
                    'alpha': current_config['alpha'],
                    'kappa': current_config['kappa'],
                    'L': current_config['L']
                }
                
                print("🤖 Requesting LLM suggestion...")
                llm_config = self.llm_interface.generate_config_suggestion(
                    current_state, 
                    self.config_memory.performance_history,
                    ENHANCED_PARAMETER_BOUNDS
                )
                
                if llm_config:
                    print(f"✅ LLM suggestion: α={llm_config['alpha']}, κ={llm_config['kappa']:.3f}, L={llm_config['L']}")
                    print(f"Operator strategy: {llm_config['operator_strategy']}")
                    
                    current_config = {
                        'alpha': llm_config['alpha'],
                        'kappa': llm_config['kappa'],
                        'L': llm_config['L'],
                        'small_value': 1e-7
                    }
                    best_operators = llm_config['operator_strategy']
                else:
                    print("⚠️ LLM failed, using adaptive strategy")
                    current_config, best_operators = self._adaptive_strategy(current_state, gap_to_target)
            else:
                print("📋 Using adaptive strategy (LLM not available)")
                current_config, best_operators = self._adaptive_strategy(current_state, gap_to_target)
        
        # Return best result if dominance not achieved
        print(f"\n⚠️ Best effort completed. Final HV: {self.best_hv:,.0f}")
        gap_remaining = target_hv - self.best_hv
        print(f"Gap remaining: {gap_remaining:,.0f}")
        
        return self.best_result, current_config, best_operators, False
    
    def _adaptive_strategy(self, current_state, gap_to_target):
        """Enhanced adaptive strategy with conservative parameter adjustment"""
        current_hv = current_state['hv']
        
        # Use best performing regions
        best_regions = self.config_memory.get_best_regions(3)
        best_operators = self.config_memory.get_best_operators(3)
        
        # Conservative parameter adjustment to avoid runtime issues
        if gap_to_target > 2000000:  # Large gap
            new_config = {
                'alpha': min(50, max(25, current_state['alpha'] + 8)),
                'kappa': min(0.20, max(0.05, current_state['kappa'] + 0.04)),
                'L': min(8, max(3, current_state['L'] + 2)),
                'small_value': 1e-7
            }
        elif gap_to_target > 1000000:  # Medium gap
            new_config = {
                'alpha': min(50, max(25, current_state['alpha'] + 5)),
                'kappa': min(0.20, max(0.05, current_state['kappa'] + 0.02)),
                'L': min(8, max(3, current_state['L'] + 1)),
                'small_value': 1e-7
            }
        else:  # Small gap - fine-tune
            new_config = {
                'alpha': min(50, max(25, current_state['alpha'] + np.random.randint(-1, 2))),
                'kappa': min(0.20, max(0.05, current_state['kappa'] + np.random.uniform(-0.005, 0.005))),
                'L': min(8, max(3, current_state['L'] + np.random.randint(-1, 2))),
                'small_value': 1e-7
            }
        
        # Select best operators
        operator_strategy = best_operators[0] if best_operators else ['swap', 'greedy_add', 'mutation']
        
        return new_config, operator_strategy

# Initialize fixed optimizer
if 'fixed_llm_interface' in locals():
    fixed_optimizer = FixedAdvancedLLMOptimizer(fixed_llm_interface, config_memory)
    print("✅ Fixed Advanced Optimizer Ready!")
else:
    print("⚠️ LLM interface not available, using fallback optimizer")
    fixed_optimizer = FixedAdvancedLLMOptimizer(None, config_memory)

✅ Fixed Advanced Optimizer Ready!


In [6]:
# Cell 5: Execute Optimized Strategy
print("\n=== EXECUTING OPTIMIZED STRATEGY ===")

# Single instance optimization with runtime controls
print("🎯 OPTIMIZED SINGLE INSTANCE OPTIMIZATION")
single_result = fixed_optimizer.optimize_instance(
    instance_file="./multiobjectives/250.2.txt",
    weights_file="./multiobjectives/Weights_2obj_FQ200.txt",
    nbitems=250,
    num_objectives=2,
    reference_hv=97640877,
    target_hv=97640877 * 1.03,
    max_iterations=8  # Reduced iterations to avoid timeout
)

# Save results
if single_result[3]:  # If dominance achieved
    result_data = single_result[0]
    np.savetxt('dominance_achieved.txt', result_data['pareto_solutions'])
    print(f"\n🎉 DOMINANCE RESULTS SAVED TO 'dominance_achieved.txt'")
else:
    result_data = single_result[0]
    np.savetxt('best_results.txt', result_data['pareto_solutions'])
    print(f"\n📊 BEST RESULTS SAVED TO 'best_results.txt'")

# Update configuration memory with final results
final_config = single_result[1]
final_operators = single_result[2]
final_hv = calculate_hypervolume_2d(result_data['pareto_solutions'])

config_memory.add_result(
    final_config,
    final_hv >= (97640877 * 1.03),
    final_hv - 97640877,
    result_data['avg_time_per_run'],
    final_hv,
    final_operators
)

print(f"\n🏆 OPTIMIZATION COMPLETE")
print(f"Final HV: {final_hv:,.0f}")
print(f"Reference HV: 97,640,877")
print(f"Target HV: 100,590,123")
print(f"Improvement: {((final_hv - 97640877) / 97640877) * 100:.2f}%")
print(f"Dominance Achieved: {'YES' if single_result[3] else 'NO'}")

# Show LLM statistics if available
if 'fixed_llm_interface' in locals() and fixed_llm_interface:
    print(f"\n🤖 LLM Status: {fixed_llm_interface.connection_status}")
    print(f"Total calls: {fixed_llm_interface.call_count}")
    print(f"Successful calls: {fixed_llm_interface.successful_calls}")
    print(f"Failed calls: {fixed_llm_interface.failed_calls}")
    print(f"Success rate: {fixed_llm_interface.successful_calls / max(1, fixed_llm_interface.call_count) * 100:.1f}%")


=== EXECUTING OPTIMIZED STRATEGY ===
🎯 OPTIMIZED SINGLE INSTANCE OPTIMIZATION

🚀 ADVANCED LLM-GUIDED OPTIMIZATION
Reference HV: 97,640,877
Target HV: 100,570,103
Gap to close: 2,929,226
LLM Status: ✅ Working

🔄 Iteration 1/8
   Configuration: 6 runs × 100 iterations, 5.0s threshold
🚀 Optimized Parameters: alpha=35, kappa=0.120, L=5, small=1.0e-07
   Operator strategy: ['swap', 'greedy_add', 'mutation']

 LLM-Enhanced MOACP: 6 runs × 100 iterations
 LLM-enhanced runtime threshold: 5.00s per run
 Reference HV: 97,640,877
 Target HV: 100,570,103
   Run 1/6... ✓ 224 solutions, 3.60s
✓ 195 solutions, 3.87s
✓ 212 solutions, 4.09s
✓ 205 solutions, 4.19s
✓ 197 solutions, 4.14s
✓ 200 solutions, 3.51s
 LLM-Enhanced Complete: 1233 total solutions, 23.40s total, 3.90s avg/run
Result HV: 98,406,624
Gap to target: 2,163,479
✓ New best result: 98,406,624
🤖 Requesting LLM suggestion...
⚠️ Error calling Ollama: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=10)
✅ LLM s

In [7]:
# Execute the fixed optimization strategy
print("\n=== EXECUTING FIXED OPTIMIZATION STRATEGY ===")

# Option 1: Single instance optimization (faster)
print("🎯 FIXED SINGLE INSTANCE OPTIMIZATION")
single_result = fixed_optimizer.optimize_instance(
    instance_file="./multiobjectives/250.2.txt",
    weights_file="./multiobjectives/Weights_2obj_FQ200.txt",
    nbitems=250,
    num_objectives=2,
    reference_hv=97640877,
    target_hv=97640877 * 1.03,
    max_iterations=10
)

# Save results
if single_result[3]:  # If dominance achieved
    result_data = single_result[0]
    np.savetxt('dominance_achieved.txt', result_data['pareto_solutions'])
    print(f"\n🎉 DOMINANCE RESULTS SAVED TO 'dominance_achieved.txt'")
else:
    result_data = single_result[0]
    np.savetxt('best_results.txt', result_data['pareto_solutions'])
    print(f"\n📊 BEST RESULTS SAVED TO 'best_results.txt'")

# Update configuration memory with final results
final_config = single_result[1]
final_operators = single_result[2]
final_hv = calculate_hypervolume_2d(result_data['pareto_solutions'])

config_memory.add_result(
    final_config,
    final_hv >= (97640877 * 1.03),
    final_hv - 97640877,
    result_data['avg_time_per_run'],
    final_hv,
    final_operators
)

print(f"\n🏆 OPTIMIZATION COMPLETE")
print(f"Final HV: {final_hv:,.0f}")
print(f"Reference HV: 97,640,877")
print(f"Target HV: 100,590,123")
print(f"Improvement: {((final_hv - 97640877) / 97640877) * 100:.2f}%")
print(f"Dominance Achieved: {'YES' if single_result[3] else 'NO'}")

# Show LLM statistics if available
if 'fixed_llm_interface' in locals() and fixed_llm_interface:
    print(f"\n🤖 LLM Status: {fixed_llm_interface.connection_status}")
    print(f"Total calls: {fixed_llm_interface.call_count}")
    print(f"Successful calls: {fixed_llm_interface.successful_calls}")
    print(f"Failed calls: {fixed_llm_interface.failed_calls}")
    print(f"Success rate: {fixed_llm_interface.successful_calls / max(1, fixed_llm_interface.call_count) * 100:.1f}%")


=== EXECUTING FIXED OPTIMIZATION STRATEGY ===
🎯 FIXED SINGLE INSTANCE OPTIMIZATION

🚀 ADVANCED LLM-GUIDED OPTIMIZATION
Reference HV: 97,640,877
Target HV: 100,570,103
Gap to close: 2,929,226
LLM Status: ✅ Working

🔄 Iteration 1/10
   Configuration: 6 runs × 100 iterations, 5.0s threshold
🚀 Optimized Parameters: alpha=35, kappa=0.120, L=5, small=1.0e-07
   Operator strategy: ['swap', 'greedy_add', 'mutation']

 LLM-Enhanced MOACP: 6 runs × 100 iterations
 LLM-enhanced runtime threshold: 5.00s per run
 Reference HV: 97,640,877
 Target HV: 100,570,103
✓ 224 solutions, 4.12s
✓ 195 solutions, 4.49s
⚠️ 212 solutions, 5.42s
✓ 205 solutions, 4.90s
✓ 197 solutions, 4.46s
✓ 200 solutions, 3.48s
 LLM-Enhanced Complete: 1233 total solutions, 26.87s total, 4.48s avg/run
Result HV: 98,406,624
Gap to target: 2,163,479
🤖 Requesting LLM suggestion...
⚠️ Error calling Ollama: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=10)
✅ LLM suggestion: α=45, κ=0.170, L=7
Operato


KeyboardInterrupt

